# Investigation 2 — Relinquishment Signal (Q2)

**Hypothesis:** Blocks adjacent to recently relinquished leases receive bids at a lower rate than similar blocks with no adjacent relinquishments.

**Method:** Flag available blocks adjacent to relinquishments in the 36-month window before the sale. Compare bid rates, controlling for water depth.

**Success:** Relinquishment-adjacent blocks bid at <= 50% the rate of non-relinquishment blocks.

---
*Prototype using Sale 247 (March 2017). Swap to Dec 2025 for final validation.*

In [ ]:
import sys, os
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency
from libpysal.weights import Rook

# Colab: ROOT = repo clone location; local: two levels up from notebooks/
try:
    import google.colab  # noqa: F401
    ROOT = "/content/block-party"
except ImportError:
    ROOT = os.path.abspath(os.path.join("..", ".."))

sys.path.insert(0, os.path.join(ROOT, "mvp0"))
import boem_loader as bl

SALE_DATE = pd.Timestamp("2017-03-22")
LOOKBACK_MONTHS = 36
LOOKBACK_START = SALE_DATE - pd.DateOffset(months=LOOKBACK_MONTHS)
print(f"Sale date: {SALE_DATE.date()}")
print(f"Relinquishment window: {LOOKBACK_START.date()} → {SALE_DATE.date()}")

## 1. Load data

In [ ]:
sales  = bl.load_master_sales(os.path.join(ROOT, "data/lease-sales/master_lease_sales.csv"))
lh     = bl.load_lease_history(os.path.join(ROOT, "data/lease-sales/cleaned_lease_history.csv"))
blocks = bl.load_blocks(os.path.join(ROOT, "data/shapefiles/blocks.shp"), to_utm=True)
wells  = bl.load_boreholes(os.path.join(ROOT, "data/wells/mv_boreholes_all.txt"), region="G", to_utm=True)

print(f"Sale bids: {len(sales)}")
print(f"Lease history: {len(lh)}")
print(f"Blocks: {len(blocks)}")
print(f"Wells (for water depth): {len(wells)}")

## 2. Build sale universe and adjacency

In [ ]:
sale_prots = set(sales["Protraction_ID"].unique())
universe = blocks[blocks["Protraction_ID"].isin(sale_prots)].copy().reset_index(drop=True)
universe["block_idx"] = range(len(universe))
print(f"Universe: {len(universe)} blocks in {len(sale_prots)} protraction areas")

# Rook adjacency
w = Rook.from_dataframe(universe, use_index=False)
adj = {i: set(w.neighbors[i]) for i in range(w.n)}
print(f"Adjacency: mean {w.mean_neighbors:.1f} neighbors")

## 3. Identify relinquishments in the lookback window

In [ ]:
relin = lh[
    (lh["Lease_Status"] == "RELINQ") &
    (lh["Status_Date"] >= LOOKBACK_START) &
    (lh["Status_Date"] <= SALE_DATE)
].copy()
print(f"Relinquishments in 36-month window: {len(relin)}")

# Join to blocks in universe
relin_blocks = relin.merge(
    universe[["AREA_CODE", "Block_Number", "block_idx"]],
    left_on=["Area_Code", "Block_Number"],
    right_on=["AREA_CODE", "Block_Number"],
    how="inner",
)
print(f"Relinquished blocks in sale universe: {relin_blocks['block_idx'].nunique()}")

relin_idxs = set(relin_blocks["block_idx"])

## 4. Flag blocks adjacent to relinquishments

In [ ]:
# Blocks adjacent to any relinquished block
relin_adj_idxs = set()
for ri in relin_idxs:
    relin_adj_idxs.update(adj.get(ri, set()))
relin_adj_idxs -= relin_idxs  # exclude the relinquished blocks themselves

universe["relin_adjacent"] = universe["block_idx"].isin(relin_adj_idxs)
print(f"Blocks adjacent to relinquishment: {universe['relin_adjacent'].sum()}")
print(f"Blocks NOT adjacent: {(~universe['relin_adjacent']).sum()}")

## 5. Mark which blocks received bids

In [ ]:
bid_pairs = set(zip(sales["Protraction_ID"], sales["Block_Number"]))
universe["did_bid"] = [
    (r["Protraction_ID"], r["Block_Number"]) in bid_pairs
    for _, r in universe.iterrows()
]
print(f"Blocks with bids: {universe['did_bid'].sum()}")

## 6. Compute bid rates: relinquishment-adjacent vs. not

In [ ]:
ct = pd.crosstab(universe["relin_adjacent"], universe["did_bid"], margins=True)
ct.index = ct.index.map({False: "No relin. adjacent", True: "Relin. adjacent", "All": "All"})
ct.columns = ct.columns.map({False: "No bid", True: "Bid", "All": "Total"})
display(ct)

rate_relin = universe.loc[universe["relin_adjacent"], "did_bid"].mean()
rate_no_relin = universe.loc[~universe["relin_adjacent"], "did_bid"].mean()
ratio = rate_relin / rate_no_relin if rate_no_relin > 0 else float("inf")

print(f"\nBid rate (relin-adjacent):     {rate_relin:.4%}")
print(f"Bid rate (no relin-adjacent):  {rate_no_relin:.4%}")
print(f"Ratio: {ratio:.2f}  (< 0.50 = cold zone confirmed)")

# Chi-square
ct_vals = pd.crosstab(universe["relin_adjacent"], universe["did_bid"])
chi2, p_val, dof, _ = chi2_contingency(ct_vals)
print(f"Chi-square: {chi2:.2f}, p = {p_val:.2e}")

## 7. Control for water depth

Estimate water depth per block from median well depth, then bucket into shelf (< 200m), deepwater (200–1500m), and ultra-deep (> 1500m).

In [ ]:
# Median water depth per (Area_Code, Block_Number) from borehole data
well_depth = (
    wells.groupby(["Area_Code", "Block_Number"])["Water_Depth"]
    .median()
    .reset_index()
    .rename(columns={"Water_Depth": "Water_Depth_ft"})
)
well_depth["Water_Depth_m"] = well_depth["Water_Depth_ft"] * 0.3048

universe = universe.merge(
    well_depth[["Area_Code", "Block_Number", "Water_Depth_m"]],
    left_on=["AREA_CODE", "Block_Number"],
    right_on=["Area_Code", "Block_Number"],
    how="left",
).drop(columns=["Area_Code"])

# Depth buckets
universe["depth_bucket"] = pd.cut(
    universe["Water_Depth_m"],
    bins=[0, 200, 1500, 99999],
    labels=["Shelf (<200m)", "Deep (200–1500m)", "Ultra-deep (>1500m)"],
)

has_depth = universe["depth_bucket"].notna()
print(f"Blocks with depth data: {has_depth.sum()} / {len(universe)}")
print(universe["depth_bucket"].value_counts().to_string())

In [ ]:
# Bid rates by depth bucket × relinquishment adjacency
depth_analysis = (
    universe[has_depth]
    .groupby(["depth_bucket", "relin_adjacent"])["did_bid"]
    .agg(["sum", "count", "mean"])
    .rename(columns={"sum": "bids", "count": "blocks", "mean": "bid_rate"})
)
print("Bid rates by depth and relinquishment adjacency:")
display(depth_analysis)

## 8. Decay analysis: 0–12 months vs. 13–36 months

In [ ]:
# Split relinquishments into recent (0-12mo) and older (13-36mo)
cutoff_12mo = SALE_DATE - pd.DateOffset(months=12)

relin_recent = relin_blocks[relin_blocks["Status_Date"] >= cutoff_12mo]
relin_older  = relin_blocks[relin_blocks["Status_Date"] < cutoff_12mo]

recent_adj = set()
for ri in set(relin_recent["block_idx"]):
    recent_adj.update(adj.get(ri, set()))

older_adj = set()
for ri in set(relin_older["block_idx"]):
    older_adj.update(adj.get(ri, set()))

universe["relin_recent"] = universe["block_idx"].isin(recent_adj - relin_idxs)
universe["relin_older"]  = universe["block_idx"].isin(older_adj - relin_idxs)

decay = pd.DataFrame({
    "Window": ["0–12 months", "13–36 months", "No relinquishment"],
    "Blocks": [
        universe["relin_recent"].sum(),
        universe["relin_older"].sum(),
        (~universe["relin_adjacent"]).sum(),
    ],
    "Bid rate": [
        universe.loc[universe["relin_recent"], "did_bid"].mean(),
        universe.loc[universe["relin_older"], "did_bid"].mean(),
        rate_no_relin,
    ],
})
print("Relinquishment decay:")
display(decay)

## 9. Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

from matplotlib.patches import Patch

# Bar chart: overall bid rates
ax = axes[0]
rates = pd.Series({"Relin. adjacent": rate_relin * 100, "Not adjacent": rate_no_relin * 100})
rates.plot.bar(ax=ax, color=["#F44336", "#4CAF50"], edgecolor="black")
ax.set_ylabel("Bid rate (%)")
ax.set_title(f"Relinquishment Effect (ratio = {ratio:.2f})")
ax.tick_params(axis="x", rotation=0)

# Decay curve
ax = axes[1]
ax.bar(decay["Window"], decay["Bid rate"] * 100, color=["#F44336", "#FF9800", "#4CAF50"], edgecolor="black")
ax.set_ylabel("Bid rate (%)")
ax.set_title("Relinquishment Signal Decay")
ax.tick_params(axis="x", rotation=15)

# Map
ax = axes[2]
universe.plot(ax=ax, color="whitesmoke", edgecolor="gray", linewidth=0.2)
relin_gdf = universe[universe["block_idx"].isin(relin_idxs)]
relin_gdf.plot(ax=ax, color="#F44336", edgecolor="black", linewidth=0.3)
adj_gdf = universe[universe["relin_adjacent"]]
adj_gdf.plot(ax=ax, color="#FFCDD2", edgecolor="gray", linewidth=0.2)
bid_blocks = universe[universe["did_bid"]]
bid_blocks.plot(ax=ax, color="#2196F3", edgecolor="black", linewidth=0.5)
ax.set_title("Relinquishments & Bids")

# Create manual legend to avoid PatchCollection warning
legend_elements = [
    Patch(facecolor="#F44336", edgecolor="black", label="Relinquished"),
    Patch(facecolor="#FFCDD2", edgecolor="gray", label="Adjacent to relin."),
    Patch(facecolor="#2196F3", edgecolor="black", label="Bid placed"),
]
ax.legend(handles=legend_elements, loc="lower left", fontsize=7)

plt.tight_layout()
plt.show()

## 10. Interpretation

| Result | Interpretation | Action |
|--------|---------------|--------|
| Relin-adjacent blocks bid at <= 50% rate | Cold zone signal confirmed | Include as MVP 1 feature |
| Marginal difference (50%–80%) | Signal exists but weak | Include, but don't feature prominently |
| No meaningful difference | Signal not present | Remove cold-zone feature from MVP 1 |

**Note:** Prototype uses Sale 247. Final validation should use Dec 2025.